In [4]:
import torch 
import torch.nn as nn 
import torchvision
import torch.optim as optim 
from torchvision.datasets import CIFAR10

In [7]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [10]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [13]:
trainset=CIFAR10(root="./data",train=True,transform=transform)
testset=CIFAR10(root="./data",train=False,transform=transform)

c:\Users\HP\anaconda3\envs\deep_learning\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [14]:
trainloader=DataLoader(trainset,batch_size=32,shuffle=True)
testloader=DataLoader(testset,batch_size=32)

### Build the CNN

In [18]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64, kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)
        return x

In [19]:
model=CNN()

In [20]:
criteria=nn.CrossEntropyLoss()

In [23]:
optimizer=optim.Adam(model.parameters())

In [25]:
epochs=10

for epoch in range(epochs):

    epoch_training_loss=0.0

    for images , labels in trainloader:

        optimizer.zero_grad()

        output=model.forward(images)

        loss=criteria(output,labels)

        loss.backward()

        optimizer.step()

        epoch_training_loss+=loss.item()

    print(f"{epoch+1}/{epochs} => training loss is {epoch_training_loss/len(trainloader)}")

1/10 => training loss is 1.0911942332201254
2/10 => training loss is 0.8197510901812324
3/10 => training loss is 0.6616590599638487
4/10 => training loss is 0.5449619652061072
5/10 => training loss is 0.4484254437536287
6/10 => training loss is 0.3623177920225638
7/10 => training loss is 0.28809343385066233
8/10 => training loss is 0.23148306685256143
9/10 => training loss is 0.18722239183708883
10/10 => training loss is 0.1527677620081821


### Evaluation

In [28]:
total=0.0
correct=0.0

model.eval()
with torch.no_grad():
    for images,labels in testloader:
        outputs=model.forward(images)

        _,predicted=torch.max(outputs,1)

        correct+=(predicted==labels).sum().item()
        total+=labels.size(0)

        accuracy=correct/total*100

print(accuracy)

75.03
